# Structural 3-Arm Dynamics — Manipulation vs. No-Manipulation Control

Split out of `structural-backfire-analysis.ipynb` (that notebook had grown too large).
Shared GEXF-parsing / metric helpers live in `bf_common.py` (imported below); this
notebook owns the one expensive step — parsing every GEXF snapshot across all three
arms — and saves the results to `results/BF_analysis/structural_global_3arm.csv` /
`structural_camp_3arm.csv` / `outcome_camp_3arm.csv`. `camp-echo-chamber-metrics.ipynb`
reads those CSVs rather than re-parsing GEXFs.

Each seed has three runs:

| Arm | folder | meaning |
|---|---|---|
| `manip` dir=+1 | `run_<s>_dir_1.0_manip_1`  | targeted hub extremized toward +1 |
| `manip` dir=−1 | `run_<s>_dir_-1.0_manip_1` | targeted hub extremized toward −1 |
| `control`      | `run_<s>_dir_1.0_manip_0`  | would-be target selected but **never flagged** (behaves normally) |

The control is **bit-identical to its manip siblings up to the onset step (`ONSET_STEP`)**
(same seed; manipulation only diverges the target from the onset). So each agent's
`opinionClass` at the onset step is the same node-for-node across all three arms, which
lets us apply the same **direction-relative camp labels** to the control and difference
it cleanly.

The would-be target id is read from `target_id.json` (written in *both* arms), so the
control — which has no GEXF `target=true` node — can still be tracked identically.

Structural resolution is 5 000 steps (GEXF cadence). We compute **global** metrics
(λ₂ on the giant component, modularity, transitivity, giant-component fraction) and
**per-camp local** metrics (within-camp density, E-I homophily index, camp-induced
λ₂ cohesion, and the target hub's follower share coming from each camp). The per-camp
collection also computes the giant-component-restricted echo-chamber metrics
(`compute_camp_giant_metrics` — modularity, clustering, path length, disagreement, etc.,
consumed by `camp-echo-chamber-metrics.ipynb`) so both live in the same saved CSV.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from bf_common import *


### 3.1 Global structure over time — manip(±) vs control

Three series per metric: manipulated toward +1, manipulated toward −1, and the
no-target control (direction-agnostic). Vertical line = manipulation onset (`ONSET_STEP`).
If the manipulation drives global fragmentation, `lambda2_gc`/`giant_frac` fall and
`modularity` rises in the manip arms but not the control.

In [ ]:
# ===========================================================================
# Collection: global + per-camp structural metrics across all three arms
#   - manip runs contribute one dir_ref (= their own manipulation sign)
#   - control runs contribute BOTH dir_refs (+1 and -1), since they share the
#     pre-onset history with both manip siblings and are direction-agnostic
#   - per-camp rows also carry the giant-component echo-chamber metrics
#     (compute_camp_giant_metrics), so camp-echo-chamber-metrics.ipynb can
#     just load structural_camp_3arm.csv without re-parsing any GEXF.
# ===========================================================================
glob_records, camp_records = [], []
all_run_dirs = sorted(glob.glob(os.path.join(RESULTS_DIR, "run_*")))
print(f"Scanning {len(all_run_dirs)} run dirs over steps {STRUCT_STEPS} ...")

for run_dir in all_run_dirs:
    info = parse_run_dirname(os.path.basename(run_dir))
    if info is None:
        continue
    seed, sign, manip = info
    if SEEDS is not None and seed not in SEEDS:
        continue
    arm = 'manip' if manip == 1 else 'control'
    dir_refs = [sign] if manip == 1 else [1.0, -1.0]
    tid = load_target_id(run_dir)

    for step in STRUCT_STEPS:
        fp = _gexf_path(run_dir, step)
        if not fp:
            continue
        G, _, agents = _parse_gexf_full(fp)
        if G is None:
            continue

        gm = compute_global_structural(G)
        glob_records.append({'seed': seed, 'arm': arm, 'sign': sign,
                             'manip': manip, 'step': step, **gm})

        for dref in dir_refs:
            camp = compute_camp_structural(G, agents, dref)
            gc   = compute_camp_giant_metrics(G, agents, dref)
            fol = target_followers_by_camp(G, tid, agents, dref)
            for g in GROUP_ORDER:
                camp_records.append({'seed': seed, 'arm': arm, 'sign': sign,
                                     'manip': manip, 'dir_ref': dref, 'step': step,
                                     'camp': g, **camp[g], **gc[g], **fol[g]})

df_glob = pd.DataFrame(glob_records)
df_camp = pd.DataFrame(camp_records)
print(f"df_glob: {df_glob.shape},  df_camp: {df_camp.shape}")
print("arms:", df_glob['arm'].value_counts().to_dict())
df_glob.to_csv(os.path.join(SUMMARY_DIR, "structural_global_3arm.csv"), index=False)
df_camp.to_csv(os.path.join(SUMMARY_DIR, "structural_camp_3arm.csv"), index=False)
print("saved structural_global_3arm.csv and structural_camp_3arm.csv")


In [ ]:
# ---------------------------------------------------------------------------
# Fig 3.1: global metrics, manip(+1)/manip(-1)/control, mean +/- SEM over seeds
# ---------------------------------------------------------------------------
GLOB_METRICS = [('lambda2_gc', 'Giant-component $\\lambda_2$'),
                ('modularity', 'Modularity $Q$'),
                ('transitivity', 'Global transitivity'),
                ('giant_frac', 'Giant-component fraction')]

def _series(df, mask, col):
    sub = df[mask]
    g = sub.groupby('step')[col].agg(['mean', 'sem'])
    return g

series_defs = [
    ('manip +1', (df_glob['arm'] == 'manip') & (df_glob['sign'] == 1.0),  '#c0392b'),
    ('manip -1', (df_glob['arm'] == 'manip') & (df_glob['sign'] == -1.0), '#e67e22'),
    ('control',  (df_glob['arm'] == 'control'),                            '#2c3e50'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (col, title) in zip(axes.ravel(), GLOB_METRICS):
    for lbl, mask, color in series_defs:
        g = _series(df_glob, mask, col)
        if g.empty:
            continue
        ax.plot(g.index, g['mean'], label=lbl, color=color, lw=2)
        ax.fill_between(g.index, g['mean'] - g['sem'], g['mean'] + g['sem'],
                        color=color, alpha=0.15)
    ax.axvline(ONSET_STEP, color='k', ls='--', lw=1, alpha=0.6)
    ax.set_title(title); ax.set_xlabel('step'); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=9)
fig.suptitle('Global network structure: manipulation vs. control', fontsize=14)
fig.tight_layout(); plt.show()


### 3.2 Per-camp local structure — the weak-target spiral

Focus on the **weak_target** camp (the weak/near camp the hypothesis predicts will
decline), with the other camps for context. For each camp we overlay the manipulated
arms (pooled over direction) against the control. The spiral hypothesis predicts, for
`weak_target` only, that after onset the manip arm shows: `tgt_follower_frac` ↓ (they
unfollow the extremizing hub), `density`/`lambda2` ↓ (the camp loses internal cohesion
once the hub bridge is gone), while the control stays flat.

In [ ]:
# ---------------------------------------------------------------------------
# Fig 3.2a: weak_target camp — manip (pooled dir) vs control
# ---------------------------------------------------------------------------
CAMP_METRICS = [('tgt_follower_frac', 'Hub follower share from camp'),
                ('density',           'Within-camp density'),
                ('camp_giant_frac',   'Within-camp giant fraction (cohesion)'),
                ('ei_index',          'E-I index (+1 hetero / -1 echo)')]

def camp_series(camp, mask, col):
    sub = df_camp[(df_camp['camp'] == camp) & mask]
    return sub.groupby('step')[col].agg(['mean', 'sem'])

def plot_camp(camp, title_suffix=''):
    m_mask = (df_camp['arm'] == 'manip')
    c_mask = (df_camp['arm'] == 'control')
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
    for ax, (col, title) in zip(axes, CAMP_METRICS):
        for lbl, mask, color in [('manip', m_mask, '#c0392b'),
                                 ('control', c_mask, '#2c3e50')]:
            g = camp_series(camp, mask, col)
            if g.empty:
                continue
            ax.plot(g.index, g['mean'], label=lbl, color=color, lw=2)
            ax.fill_between(g.index, g['mean'] - g['sem'], g['mean'] + g['sem'],
                            color=color, alpha=0.15)
        ax.axvline(ONSET_STEP, color='k', ls='--', lw=1, alpha=0.6)
        ax.set_title(title); ax.set_xlabel('step'); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=9)
    fig.suptitle(f'Camp = {camp} {title_suffix}', fontsize=14)
    fig.tight_layout(); plt.show()

plot_camp('strong_opposite', '(extreme opposite-side camp — never followed the hub)')
plot_camp('weak_opposite',   '(moderate opposite-side camp)')
plot_camp('neutral',         '(centre camp)')
plot_camp('weak_target',     '(weak/near camp — hypothesis focus)')
plot_camp('strong_target',   '(extreme same-side camp — should follow the hub)')


In [ ]:
# ---------------------------------------------------------------------------
# Fig 3.2b: full camp x metric grid (manip vs control), all five camps
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(len(CAMP_METRICS), len(GROUP_ORDER),
                         figsize=(4.2 * len(GROUP_ORDER), 3.4 * len(CAMP_METRICS)),
                         sharex=True)
for r, (col, title) in enumerate(CAMP_METRICS):
    for c, camp in enumerate(GROUP_ORDER):
        ax = axes[r, c]
        for lbl, mask, color in [('manip', df_camp['arm'] == 'manip', '#c0392b'),
                                 ('control', df_camp['arm'] == 'control', '#2c3e50')]:
            g = camp_series(camp, mask, col)
            if g.empty:
                continue
            ax.plot(g.index, g['mean'], color=color, lw=1.6, label=lbl)
            ax.fill_between(g.index, g['mean'] - g['sem'], g['mean'] + g['sem'],
                            color=color, alpha=0.13)
        ax.axvline(ONSET_STEP, color='k', ls='--', lw=0.8, alpha=0.5)
        ax.grid(alpha=0.3)
        if r == 0:
            ax.set_title(camp, fontsize=11, color=CAMP_COLORS.get(camp, 'k'))
        if c == 0:
            ax.set_ylabel(title, fontsize=10)
axes[0, 0].legend(fontsize=8)
fig.suptitle('Per-camp local structure across steps: manip (red) vs control (black)',
             fontsize=14)
fig.tight_layout(); plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Table 3.3: onset->end change (step ONSET_STEP -> 40000) by camp and arm.
# delta = end - onset, averaged over the seed(xdirection) units. The
# manip - control column isolates the manipulation-attributable structural
# change per camp.
# ---------------------------------------------------------------------------
def onset_end_delta(col):
    piv = (df_camp[df_camp['step'].isin([ONSET_STEP, 40000])]
           .pivot_table(index=['arm', 'camp'], columns='step', values=col, aggfunc='mean'))
    piv['delta'] = piv.get(40000) - piv.get(ONSET_STEP)
    out = piv['delta'].unstack('arm')
    if 'manip' in out and 'control' in out:
        out['manip - control'] = out['manip'] - out['control']
    return out.reindex(GROUP_ORDER)

for col, title in CAMP_METRICS:
    print(f"\n=== {title}  ({col}): onset->end delta ===")
    print(onset_end_delta(col).round(4))


### 3.3 Outcome: posting probability & comfort rate — manip vs control

Section 3.1/3.2 are purely structural (follow-topology). This closes the loop to the
actual hypothesis: does **posting behaviour** in each camp diverge from the no-target
counterfactual, and is weak_target's divergence bigger / earlier than other camps?

Uses `postProbMean`/`cRateMean` from `metrics/result_*.csv` (100-step resolution, finer
than the 5000-step GEXF cadence used above). Same manip/control overlay convention as
3.1/3.2; control pools both `dir_ref` values since it is direction-agnostic.

In [ ]:
# ===========================================================================
# Collection: postProb & cRate per camp, manip vs control (100-step)
# ===========================================================================
outcome_frames = []
for run_dir in all_run_dirs:
    info = parse_run_dirname(os.path.basename(run_dir))
    if info is None:
        continue
    seed, sign, manip = info
    if SEEDS is not None and seed not in SEEDS:
        continue
    arm = 'manip' if manip == 1 else 'control'
    dir_refs = [sign] if manip == 1 else [1.0, -1.0]

    for dref in dir_refs:
        df_pp = load_metric_timeseries(run_dir, dref, "postProbMean")
        df_cr = load_metric_timeseries(run_dir, dref, "cRateMean")
        if df_pp is None or df_cr is None:
            continue
        pp_long = df_pp.melt(id_vars='step', value_vars=GROUP_ORDER,
                             var_name='camp', value_name='postprob')
        cr_long = df_cr.melt(id_vars='step', value_vars=GROUP_ORDER,
                             var_name='camp', value_name='crate')
        merged = pp_long.merge(cr_long, on=['step', 'camp'])
        merged['seed'], merged['arm'], merged['dir_ref'] = seed, arm, dref
        outcome_frames.append(merged)

df_outcome = pd.concat(outcome_frames, ignore_index=True)
print(f"df_outcome: {df_outcome.shape}")
print("arms:", df_outcome[['seed', 'arm']].drop_duplicates()['arm'].value_counts().to_dict())
df_outcome.to_csv(os.path.join(SUMMARY_DIR, "outcome_camp_3arm.csv"), index=False)
print("saved outcome_camp_3arm.csv")


In [ ]:
# ---------------------------------------------------------------------------
# Fig 3.3: postProb & cRate, manip vs control, per camp
# ---------------------------------------------------------------------------
def outcome_series(camp, arm, col):
    sub = df_outcome[(df_outcome['camp'] == camp) & (df_outcome['arm'] == arm)]
    return sub.groupby('step')[col].agg(['mean', 'sem'])

def plot_outcome(camp, title_suffix=''):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for ax, (col, title) in zip(axes, [('postprob', 'Mean postProb'),
                                       ('crate', 'Mean cRate (comfort rate)')]):
        for lbl, color in [('manip', '#c0392b'), ('control', '#2c3e50')]:
            g = outcome_series(camp, lbl, col)
            if g.empty:
                continue
            ax.plot(g.index, g['mean'], label=lbl, color=color, lw=1.8)
            ax.fill_between(g.index, g['mean'] - g['sem'], g['mean'] + g['sem'],
                            color=color, alpha=0.15)
        ax.axvline(ONSET_STEP, color='k', ls='--', lw=1, alpha=0.6)
        ax.set_title(title); ax.set_xlabel('step'); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=9)
    fig.suptitle(f'Camp = {camp} {title_suffix}', fontsize=13,
                color=CAMP_COLORS.get(camp, 'k'))
    fig.tight_layout(); plt.show()

for camp, suffix in [('strong_opposite', '(extreme opposite-side camp)'),
                     ('weak_opposite',   '(moderate opposite-side camp)'),
                     ('neutral',         '(centre camp)'),
                     ('weak_target',     '(weak/near camp — hypothesis focus)'),
                     ('strong_target',   '(extreme same-side camp)')]:
    plot_outcome(camp, suffix)


In [ ]:
# ---------------------------------------------------------------------------
# Table 3.4: onset->end change in postProb/cRate by camp and arm
# ---------------------------------------------------------------------------
def outcome_delta_table(col):
    piv = (df_outcome[df_outcome['step'].isin([ONSET_STEP, 40000])]
           .pivot_table(index=['arm', 'camp'], columns='step', values=col, aggfunc='mean'))
    piv['delta'] = piv.get(40000) - piv.get(ONSET_STEP)
    out = piv['delta'].unstack('arm').reindex(GROUP_ORDER)
    if 'manip' in out and 'control' in out:
        out['manip - control'] = out['manip'] - out['control']
    return out.round(4)

for col, title in [('postprob', 'Mean postProb'), ('crate', 'Mean cRate')]:
    print(f"\n=== {title} ({col}): onset->end delta ===")
    print(outcome_delta_table(col))
